Project 2: Independent feasibility checker

Run from a folder that contains data/ (the workbooks) and results/ (the plans written by the model notebook).

In [1]:
   import os, sys


Helpers used only by the tests

The tests below use these functions to build realistic broken plans. The checker itself (next section) does not use them and shares no code with the model.

In [2]:
"""
MODULE 1 - DATA AND TIME (owner: Micah)

Reads the workbook, puts every time on one clock (absolute hours = 24 x day + hour),
and works out each shipment's timeline from its ready time:
loading start, loaded departure, destination service start, release.

Rules handled here: R3 (loading start), R4 (loaded trip, release, a late ready time
pushes the release later), R5 (finish before the horizon), and the opening-hours
logic used by R12.
"""
import math
import pandas as pd

HOOK = 0.25   # hours to hook a trailer   (parameters sheet: hook_hours)
DROP = 0.25   # hours to drop a trailer   (parameters sheet: drop_hours)


# ---------------------------------------------------------------------------
# 1. Read the workbook
# ---------------------------------------------------------------------------
def load_instance(path):
    """Read the 7 input sheets and return one dictionary with everything."""
    wb = pd.read_excel(path, sheet_name=None)
    par = wb["parameters"].set_index("parameter")["value"]
    inst = {
        "facilities": wb["facilities"].set_index("facility_id"),
        "depots": wb["depots"].set_index("depot_id"),
        "tractors": wb["tractors"].set_index("tractor_id"),
        "shipments": wb["shipments"].set_index("shipment_id"),
        "trailers": wb["trailer_state"].set_index("trailer_id"),
        "H": 24 * int(par["horizon_end_day"]),          # horizon end, absolute hours
        "demand_days": int(par["demand_days"]),
        "late_allowed": int(par.get("late_allowed", 0)) == 1,
        "excess_allowed": int(par.get("excess_allowed", 0)) == 1,
    }
    assert float(par["hook_hours"]) == HOOK and float(par["drop_hours"]) == DROP
    # lanes: (origin, destination) -> (km, hours). Direction matters.
    inst["lanes"] = {
        (r.origin_id, r.destination_id): (float(r.distance_km), float(r.drive_hours))
        for r in wb["lanes"].itertuples()
    }
    inst["F"] = list(inst["facilities"].index)
    inst["is_main"] = inst["excess_allowed"]           # main file: penalties allowed
    return inst


# ---------------------------------------------------------------------------
# 2. Time helpers
# ---------------------------------------------------------------------------
def abs_time(day, hour):
    """Absolute time = 24 x day + hour. Day 1 at 10:00 -> 34.0"""
    return 24 * day + hour


def lane(inst, a, b):
    """(km, hours) from a to b. Same place = 0 km, 0 hours."""
    if a == b:
        return 0.0, 0.0
    return inst["lanes"][(a, b)]


def opening_hours(inst, site):
    """Daily opening interval (open, close) of a facility or a depot."""
    row = inst["facilities"].loc[site] if site in inst["facilities"].index else inst["depots"].loc[site]
    return float(row["open_hour"]), float(row["close_hour"])


def next_open(inst, site, t, duration):
    """Earliest s >= t when `site` is open long enough for a task of `duration` hours
    (finishing exactly at closing is allowed; 0-24 means always open)."""
    o, c = opening_hours(inst, site)
    if o == 0 and c == 24:
        return t
    day = math.floor(t / 24)
    for d in range(day, day + 5):
        start = max(t, 24 * d + o)
        if start + duration <= 24 * d + c + 1e-9:
            return start
    return None


# ---------------------------------------------------------------------------
# 3. A shipment's timeline from a given ready time
# ---------------------------------------------------------------------------
def shipment_times(inst, j, ready):
    """Everything that follows once the empty for shipment j is ready at `ready`.
    Loading starts at the first open moment at/after max(deadline, ready) that fits
    loading + hook (R3). A late ready time moves every later step, including the
    release at the destination (R4)."""
    s = inst["shipments"].loc[j]
    deadline = abs_time(s["day"], s["deadline_hour"])
    load_start = next_open(inst, s["origin_id"], max(deadline, ready), s["loading_hours"] + HOOK)
    loaded_depart = load_start + s["loading_hours"] + HOOK
    arrive_dest = loaded_depart + lane(inst, s["origin_id"], s["destination_id"])[1]
    service_start = next_open(inst, s["destination_id"], arrive_dest, DROP + s["unloading_hours"])
    release = service_start + DROP + s["unloading_hours"]
    return {"ready": ready, "late_hours": max(0.0, ready - deadline),
            "load_start": load_start, "loaded_depart": loaded_depart,
            "arrive_dest": arrive_dest, "service_start": service_start, "release": release}


def shipment_table(inst):
    """One row per shipment with its on-time timeline (ready = deadline)."""
    rows = []
    for j, s in inst["shipments"].iterrows():
        deadline = abs_time(s["day"], s["deadline_hour"])
        t = shipment_times(inst, j, deadline)
        if t["release"] > inst["H"]:
            raise ValueError(f"{j} cannot finish before the horizon (R5)")
        rows.append({"shipment_id": j, "origin": s["origin_id"], "destination": s["destination_id"],
                     "deadline": deadline, **t,
                     "late_rate": float(s["late_penalty_cad_per_hour"]),
                     "loaded_km": lane(inst, s["origin_id"], s["destination_id"])[0],
                     "loaded_cost": float(s["loaded_service_cost_cad"])})
    return pd.DataFrame(rows).set_index("shipment_id")



Helpers used only by the tests

The tests below use these functions to build realistic broken plans. The checker itself (next section) does not use them and shares no code with the model.

In [3]:
"""
MODULE 2 - TOUR BUILDER (owner: Henry)

Builds the candidate set R: every legal tractor day, with exact times and cost.
  1-move tour: depot -> f -> hook -> tow to g -> drop -> depot
  2-move tour:         depot -> f1 -> hook -> tow to g1 -> drop -> (bobtail to f2, or stay
                       at g1 when f2 = g1: a "chain") -> hook -> tow to g2 -> drop -> depot
                       (brief: the tractor may bobtail "from its previous drop" before it hooks)
Each tour becomes one yes/no variable z_r in the model, and its cost c_r is that
variable's coefficient in the objective.

Rules checked here, before Gurobi sees a tour: R10 (starts and ends at the home
depot), R11 shift + depot hours, R12 facility hours (waiting outside counts as duty),
R13 driving and duty limits, R14 one trailer at a time, hook/drop 0.25 h, lanes by
ordered pair, R15 depots tractor-only, R16 cost at this tractor's own rates.
"""
import pandas as pd


# How candidates are limited (the brief asks us to state this).
SETTINGS = {
    # simplified: every departure every 15 min, every pair, every 2-move tour
    "simplified": {"step": 0.25, "k_nearest_sources": None, "tractors_per_option": None,
                   "chains": True, "chain_next": None, "chain_step": 0.25},
    # main: hourly departures, 6 nearest sources per destination, 3 cheapest tractors per
    # identical tour; 2-move tours: the second drop is at one of the 2 nearest yards to the
    # first drop (nearest by lane distance in either direction, since lanes are not symmetric), the second pickup is one of the 6 nearest sources of that yard or the first
    # drop yard itself (chain); departures every 2 hours
    "main":       {"step": 1.00, "k_nearest_sources": 6, "tractors_per_option": 3,
                   "chains": True, "chain_next": 2, "chain_step": 2.0},
}


def facility_pairs(inst, k_nearest):
    """(source, destination) pairs to try. None = all ordered pairs;
    k = for each destination, its k nearest sources by lane distance."""
    F = inst["F"]
    if k_nearest is None:
        return [(f, g) for f in F for g in F if f != g]
    pairs = []
    for g in F:
        srcs = sorted((f for f in F if f != g), key=lambda f: (lane(inst, f, g)[0], f))
        pairs += [(f, g) for f in srcs[:k_nearest]]
    return pairs


def follow_tour(inst, row, day, depart, legs):
    """Follow one tractor day. legs = [(f, g)] or [(f1, g1), (f2, g2)] (f2 = g1 is a chain).
    Returns (moves, return_time, drive, bobtail_km, towing_km) or None if a rule breaks."""
    depot = row["home_depot_id"]
    _, d_close = opening_hours(inst, depot)
    latest_return = 24 * day + min(row["shift_end_hour"], d_close)       # R11
    pos, t, drive, bob, tow, moves = depot, depart, 0.0, 0.0, 0.0, []
    for f, g in legs:
        km, hr = lane(inst, pos, f)                 # bobtail to f (0 if already there)
        bob += km; drive += hr; t += hr
        hook = next_open(inst, f, t, HOOK)          # wait outside if closed     R12
        if hook is None:
            return None
        km, hr = lane(inst, f, g)                   # tow f -> g                 R14
        tow += km; drive += hr
        drop = next_open(inst, g, hook + HOOK + hr, DROP)                   # R12
        if drop is None:
            return None
        moves.append((f, g, hook, drop, drop + DROP))
        pos, t = g, drop + DROP
    km, hr = lane(inst, pos, depot)                 # bobtail home               R10, R15
    bob += km; drive += hr
    ret = t + hr
    if ret > latest_return + 1e-9:                  # R11
        return None
    if drive > row["max_drive_hours"] + 1e-9:       # R13
        return None
    if ret - depart > row["max_duty_hours"] + 1e-9: # R13 (waiting counts as duty)
        return None
    return moves, ret, drive, bob, tow


def price(row, depart, ret, bob, tow, n_moves):
    """R16: cost at this tractor's own rates."""
    c = {"cost_bobtail": bob * row["bobtail_cad_per_km"],
         "cost_towing": tow * row["towing_cad_per_km"],
         "cost_duty": (ret - depart) * row["duty_cad_per_hour"],
         "cost_fees": n_moves * row["move_fee_cad"]}
    c["cost"] = sum(c.values())
    return c


def build_tours(inst, settings):
    """Return (tours, moves): one row per candidate tour, one row per move inside it."""
    pairs = facility_pairs(inst, settings["k_nearest_sources"])
    F = inst["F"]                                    # chain: allowed second destinations from g1
    # "nearest" second yards: lanes are not symmetric, so take the n nearest by distance
    # from g1 AND the n nearest by distance to g1 (union)
    out_ = {g1: sorted((g for g in F if g != g1), key=lambda g: (lane(inst, g1, g)[0], g)) for g1 in F}
    in_ = {g1: sorted((g for g in F if g != g1), key=lambda g: (lane(inst, g, g1)[0], g)) for g1 in F}
    n2 = settings["chain_next"]
    next_dest = {g1: sorted(set(out_[g1][:n2]) | set(in_[g1][:n2])) if n2 is not None else out_[g1]
                 for g1 in F}
    # second move after a drop at g1: to g2 in next_dest[g1], picking up at g1 itself (chain)
    # or at any allowed source of g2 (bobtail from g1 to that source first)
    sources_of = {g: [] for g in F}
    for f, g in pairs:
        sources_of[g].append(f)
    second = {g1: sorted({(g1, g2) for g2 in next_dest[g1]}
                         | {(f2, g2) for g2 in next_dest[g1] for f2 in sources_of[g2]})
              for g1 in F}
    seqs2 = [((f, g), mv2) for f, g in pairs for mv2 in second[g]]
    # simplified: tours after the last demand day cannot help (no excess cost);
    # main: later days can still clear over-full yards
    last_day = inst["demand_days"] if not inst["is_main"] else inst["H"] // 24 - 1

    rows = []
    for k, row in inst["tractors"].iterrows():
        d_open, d_close = opening_hours(inst, row["home_depot_id"])
        dep = row["home_depot_id"]          # quick screen: 2-move tours whose driving alone is too long
        def drive2(s):
            (f1, g1), (f2, g2) = s
            return sum(lane(inst, a, b)[1] for a, b in [(dep, f1), (f1, g1), (g1, f2), (f2, g2), (g2, dep)])
        my_seqs2 = [list(s) for s in seqs2 if drive2(s) <= row["max_drive_hours"] + 1e-9]
        for day in range(last_day + 1):
            depart = 24 * day + max(row["shift_start_hour"], d_open)       # R11
            first = depart
            latest = 24 * day + min(row["shift_end_hour"], d_close)
            while depart < latest:
                chain_ok = settings["chains"] and abs(((depart - first) / settings["chain_step"]) % 1) < 1e-9
                for f, g in pairs:
                    r1 = follow_tour(inst, row, day, depart, [(f, g)])
                    if r1 is not None:
                        rows.append((k, row, day, depart, r1))
                if chain_ok:
                    for legs in my_seqs2:
                        r2 = follow_tour(inst, row, day, depart, legs)
                        if r2 is not None:
                            rows.append((k, row, day, depart, r2))
                depart += settings["step"]

    recs = []
    for k, row, day, depart, (mv, ret, drive, bob, tow) in rows:
        key = tuple(x for m in mv for x in (m[0], m[1], m[2], m[3]))
        recs.append({"tractor": k, "driver": row["driver_name"], "depot": row["home_depot_id"],
                     "day": day, "depart": depart, "return": ret, "drive_hours": drive,
                     "duty_hours": ret - depart, "bobtail_km": bob, "towing_km": tow,
                     "n_moves": len(mv), "_key": key, "_moves": mv,
                     **price(row, depart, ret, bob, tow, len(mv))})
    df = pd.DataFrame(recs)
    # same tractor, day and same hook/drop times = same tour; keep the cheaper (less waiting)
    df = df.sort_values("cost").drop_duplicates(["tractor", "day", "_key"])
    # main: for each depot, day and identical tour, keep the n cheapest tractors
    n = settings["tractors_per_option"]
    if n is not None:
        df = df.sort_values("cost").groupby(["depot", "day", "_key"], sort=False).head(n)
    df = df.sort_values(["day", "tractor", "depart", "_key"]).reset_index(drop=True)
    df.index = [f"R{i:06d}" for i in range(len(df))]
    df.index.name = "tour_id"

    mrows = []
    for r, mv in df["_moves"].items():
        for i, (f, g, hook, drop, drop_end) in enumerate(mv, start=1):
            mrows.append({"move_id": f"{r}-{i}", "tour_id": r, "seq": i, "from": f, "to": g,
                          "hook": hook, "drop_start": drop, "drop_end": drop_end})
    moves = pd.DataFrame(mrows).set_index("move_id")
    tours = df.drop(columns=["_key", "_moves"])
    return tours, moves


def readme_check(inst):
    """README example: T01 leaves D01 at 06:00 on Day 0, moves an empty F05 -> F01. Must cost CAD 1,030."""
    row = inst["tractors"].loc["T01"]
    mv, ret, drive, bob, tow = follow_tour(inst, row, 0, 6.0, [("F05", "F01")])
    c = price(row, 6.0, ret, bob, tow, 1)["cost"]
    assert abs(c - 1030) < 1e-6 and (mv[0][2], mv[0][4], ret) == (8.5, 12.25, 15.0), (mv, ret, c)
    return c



The checker

Reads the workbook and an exported plan, and checks everything again without trusting Gurobi. It rebuilds each tractor’s timeline (shift, depot and facility hours, driving and duty limits, hook/drop times, one shift per day), each trailer’s location over time, and the usable, physical and loaded counts at every yard. It recomputes each shipment’s ready time, loading, loaded trip and release, and the cost by component. On the main file it separates permitted, correctly charged late hours and excess-empty hours from actions that are still forbidden, such as exceeding yard capacity.

In [4]:
"""
INDEPENDENT FEASIBILITY CHECKER (owner: Sabina)

Reads only the workbook and an exported plan. It imports nothing from the model,
the tour builder or the baseline, and it never looks at a solver status. From the
plan it rebuilds:
  - each tractor's timeline (shift, depot hours, opening hours, driving, duty, one
    shift per day, one trailer at a time, home depot)
  - each trailer's location over time (hooked only where it is and once it is free;
    never in two activities at once)
  - the ready time, loading, loaded trip and release of every shipment, recomputed
    from the actual ready time
  - the three yard counts every quarter hour: usable empties, physical empties
    (allowance) and occupancy (capacity)
  - the cost by component, compared with what the plan claims
Main file: late hours and excess empty-trailer hours are PERMITTED if charged
correctly; anything else is a VIOLATION (over capacity, a trailer used before it is
there, a broken shift, a wrong charge or a release that was not moved).

Run: python code/checker.py data/project2_main.xlsx results/plan_main.xlsx
"""
import sys, math, time
from collections import defaultdict
import numpy as np
import pandas as pd

EPS = 1e-6


class Workbook:
    def __init__(self, path):
        wb = pd.read_excel(path, sheet_name=None)
        par = wb["parameters"].set_index("parameter")["value"]
        self.fac = wb["facilities"].set_index("facility_id")
        self.dep = wb["depots"].set_index("depot_id")
        self.trk = wb["tractors"].set_index("tractor_id")
        self.shp = wb["shipments"].set_index("shipment_id")
        self.trl = wb["trailer_state"].set_index("trailer_id")
        self.H = 24 * int(par["horizon_end_day"])
        self.hook, self.drop = float(par["hook_hours"]), float(par["drop_hours"])
        self.late_ok = int(par.get("late_allowed", 0)) == 1
        self.excess_ok = int(par.get("excess_allowed", 0)) == 1
        self.lanes = {(r.origin_id, r.destination_id): (float(r.distance_km), float(r.drive_hours))
                      for r in wb["lanes"].itertuples()}

    def lane(self, a, b):
        return (0.0, 0.0) if a == b else self.lanes[(a, b)]

    def hours(self, site):
        r = self.fac.loc[site] if site in self.fac.index else self.dep.loc[site]
        return float(r.open_hour), float(r.close_hour)

    def inside_open(self, site, t, dur):
        """Is [t, t + dur] inside one opening interval? (ending exactly at closing is fine)"""
        o, c = self.hours(site)
        if o == 0 and c == 24:
            return True
        d = math.floor(t / 24 + EPS)
        return 24 * d + o - EPS <= t and t + dur <= 24 * d + c + EPS

    def first_open(self, site, t, dur):
        """Earliest moment >= t at which [s, s + dur] fits an opening interval (quarter-hour scan)."""
        s = t
        for _ in range(24 * 4 * 5):
            if self.inside_open(site, s, dur):
                return s
            s = math.floor(s * 4 + 1 + EPS) / 4
        return None


def check(data_path, plan_path):
    t_start = time.time()
    wb = Workbook(data_path)
    plan = pd.read_excel(plan_path, sheet_name=None)
    tours = plan["tours"].set_index("tour_id")
    moves = plan["moves"].set_index("move_id")
    ships = plan["shipments"].set_index("shipment_id")
    claim = plan["summary"].set_index(plan["summary"].columns[0])["value"]
    V, permitted = [], []                    # violations, permitted-and-charged items
    cost = defaultdict(float)
    km = defaultdict(float)

    # ---------------- 1. tractors ----------------
    for (k, d), grp in tours.groupby(["tractor", "day"]):
        if len(grp) > 1:
            V.append(f"tractor {k} double-booked: {len(grp)} shifts on day {d}")
    for r, tr in tours.iterrows():
        k = tr["tractor"]
        if k not in wb.trk.index:
            V.append(f"{r}: unknown tractor {k}"); continue
        row = wb.trk.loc[k]; depot = row["home_depot_id"]; day = int(tr["day"])
        if tr.get("depot", depot) != depot:
            V.append(f"{r}: tour does not start at {k}'s home depot {depot}")
        do, dc = wb.hours(depot)
        start = 24 * day + max(row["shift_start_hour"], do)
        end = 24 * day + min(row["shift_end_hour"], dc)
        if tr["depart"] < start - EPS:
            V.append(f"{r}: {k} leaves at {tr['depart']} before its shift/depot opens ({start})")
        mv = moves[moves["tour_id"] == r].sort_values("seq")
        pos, t, drive, bob, tow, holding = depot, float(tr["depart"]), 0.0, 0.0, 0.0, False
        for m, x in mv.iterrows():
            kmh = wb.lane(pos, x["from"]); bob += kmh[0]; drive += kmh[1]; t += kmh[1]
            if x["hook"] < t - EPS:
                V.append(f"{m}: hook at {x['hook']} before {k} can reach {x['from']} ({t})")
            if not wb.inside_open(x["from"], x["hook"], wb.hook):
                V.append(f"{m}: hook at {x['hook']} while {x['from']} is closed")
            kmh = wb.lane(x["from"], x["to"]); tow += kmh[0]; drive += kmh[1]
            arrive = x["hook"] + wb.hook + kmh[1]
            if x["drop_start"] < arrive - EPS:
                V.append(f"{m}: drop at {x['drop_start']} before arrival at {x['to']} ({arrive})")
            if not wb.inside_open(x["to"], x["drop_start"], wb.drop):
                V.append(f"{m}: drop at {x['drop_start']} while {x['to']} is closed")
            if abs(x["drop_end"] - x["drop_start"] - wb.drop) > EPS:
                V.append(f"{m}: drop does not take {wb.drop} h")
            if x["from"] not in wb.fac.index or x["to"] not in wb.fac.index:
                V.append(f"{m}: trailers can only move between facilities")
            pos, t = x["to"], float(x["drop_end"])
        kmh = wb.lane(pos, depot); bob += kmh[0]; drive += kmh[1]
        if tr["return"] < t + kmh[1] - EPS:
            V.append(f"{r}: returns at {tr['return']} before it can reach {depot} ({t + kmh[1]})")
        if tr["return"] > end + EPS:
            V.append(f"{r}: {k} returns at {tr['return']} after shift end / depot close ({end})")
        if drive > row["max_drive_hours"] + EPS:
            V.append(f"{r}: {k} drives {drive} h > {row['max_drive_hours']}")
        duty = tr["return"] - tr["depart"]
        if duty > row["max_duty_hours"] + EPS:
            V.append(f"{r}: {k} on duty {duty} h > {row['max_duty_hours']}")
        cost["bobtail"] += bob * row["bobtail_cad_per_km"]
        cost["empty_towing"] += tow * row["towing_cad_per_km"]
        cost["duty_time"] += duty * row["duty_cad_per_hour"]
        cost["move_fees"] += len(mv) * row["move_fee_cad"]
        km["bobtail"] += bob; km["towing"] += tow

    # ---------------- 2. shipments: recompute every time from the actual ready time ----------------
    stime = {}
    for j, s in wb.shp.iterrows():
        if j not in ships.index:
            V.append(f"{j}: not served"); continue
        p = ships.loc[j]
        D = 24 * s["day"] + s["deadline_hour"]
        if p["covered_by"] != "own stock":
            if p["covered_by"] not in moves.index:
                V.append(f"{j}: covered by unknown move {p['covered_by']}"); continue
            mvr = moves.loc[p["covered_by"]]
            if mvr["to"] != s["origin_id"] or mvr["trailer"] != p["trailer"]:
                V.append(f"{j}: move {p['covered_by']} does not bring trailer {p['trailer']} to {s['origin_id']}")
            ready = float(mvr["drop_end"])                 # actual ready time = drop finished
        else:
            ready = float(p["ready"])                      # reserved at its ready time
        if abs(ready - p["ready"]) > EPS:
            V.append(f"{j}: plan says ready {p['ready']}, actual ready {ready}")
        late = max(0.0, ready - D)
        if late > EPS:
            if not wb.late_ok:
                V.append(f"{j}: missed deadline, ready at {ready} > deadline {D}")
            else:
                permitted.append(f"{j}: {late:.2f} late hours charged at {s['late_penalty_cad_per_hour']}/h")
        cost["late_penalty"] += late * s["late_penalty_cad_per_hour"]
        ls = wb.first_open(s["origin_id"], max(D, ready), s["loading_hours"] + wb.hook)
        dep = ls + s["loading_hours"] + wb.hook
        arr = dep + wb.lane(s["origin_id"], s["destination_id"])[1]
        ss = wb.first_open(s["destination_id"], arr, wb.drop + s["unloading_hours"])
        rel = ss + wb.drop + s["unloading_hours"]
        if rel > wb.H + EPS:
            V.append(f"{j}: finishes at {rel}, after the horizon {wb.H}")
        for col, val in (("load_start", ls), ("loaded_depart", dep), ("service_start", ss), ("release", rel)):
            if abs(p[col] - val) > EPS:
                V.append(f"{j}: {col} should be {val} given ready time {ready}, plan says {p[col]}")
        if abs(p["late_hours"] - late) > EPS or abs(p["late_penalty"] - late * s["late_penalty_cad_per_hour"]) > 0.01:
            V.append(f"{j}: late hours/penalty misreported ({p['late_hours']}, {p['late_penalty']}) vs ({late}, "
                     f"{late * s['late_penalty_cad_per_hour']})")
        cost["loaded_service"] += s["loaded_service_cost_cad"]
        stime[j] = (ready, ls, dep, ss, rel, s["origin_id"], s["destination_id"])

    # ---------------- 3. trailers over time ----------------
    NT = int(round(4 * wb.H))
    phys = {g: np.zeros(NT) for g in wb.fac.index}       # physical empties
    load = {g: np.zeros(NT) for g in wb.fac.index}       # loaded trailers (being loaded / unloaded)
    usable = {g: np.zeros(NT) for g in wb.fac.index}     # usable, unpromised empties
    def add(arr, g, t0, t1, d=1):
        a, b = int(round(4 * t0)), min(int(round(4 * t1)), NT)
        if b > a: arr[g][a:b] += d
    ev = defaultdict(list)                                # trailer -> events
    for m, x in moves.iterrows():
        ev[x["trailer"]] += [(x["hook"], 1, "hook", m), (x["drop_end"], 0, "drop", m)]
    for j, p in ships.iterrows():
        if j in stime:
            ev[p["trailer"]] += [(stime[j][0], 1, "use", j), (stime[j][4], 0, "release", j)]
    for tr in set(ev) - set(wb.trl.index):
        V.append(f"unknown trailer {tr}")
    for tr, row in wb.trl.iterrows():
        site, since, state = row["facility_id"], float(row["available_from"]), "free"
        pstart = since                                    # physically here since
        for t, _, kind, ref in sorted(ev.get(tr, []), key=lambda e: (e[0], e[1])):
            if kind == "hook":
                x = moves.loc[ref]
                if state != "free" or site != x["from"] or since > t + EPS:
                    V.append(f"{ref}: trailer {tr} hooked at {x['from']} at {t}, but it is "
                             f"{state} at {site} (usable from {since})")
                add(phys, site, pstart, t + wb.hook); add(usable, site, since, t)
                state, site = "moving", None
            elif kind == "drop":
                x = moves.loc[ref]
                site, since, state, pstart = x["to"], t, "free", x["drop_start"]
            elif kind == "use":
                ready, ls, dep, ss, rel, o, d = stime[ref]
                if state != "free" or site != o or since > ready + EPS:
                    V.append(f"{ref}: trailer {tr} must be free at {o} by {ready}, but it is {state} at "
                             f"{site} (usable from {since})")
                add(phys, o, pstart, ls); add(usable, o, since, min(ready, ls))
                add(load, o, ls, dep); add(load, d, ss, rel)
                state, site = "loaded", None
            elif kind == "release":
                ready, ls, dep, ss, rel, o, d = stime[ref]
                site, since, state, pstart = d, rel, "free", rel
        if state == "free":
            add(phys, site, pstart, wb.H); add(usable, site, since, wb.H)
        elif state == "moving":
            V.append(f"trailer {tr} hooked but never dropped")

    # ---------------- 4. yard counts every quarter hour ----------------
    excess_h = 0.0
    for g in wb.fac.index:
        cap, allow, beta = wb.fac.at[g, "trailer_capacity"], wb.fac.at[g, "empty_allowance"], \
            wb.fac.at[g, "excess_penalty_cad_per_trailer_hour"]
        occ = phys[g] + load[g]
        bad = np.where(occ > cap + EPS)[0]
        if len(bad):
            V.append(f"{g}: yard holds {int(occ[bad[0]])} trailers at {bad[0] / 4} > capacity {cap}"
                     f" ({len(bad)} quarter hours over)")
        over = np.clip(phys[g] - allow, 0, None)
        if over.sum() > EPS:
            if not wb.excess_ok:
                V.append(f"{g}: {int(phys[g].max())} physical empties > allowance {allow}")
            else:
                excess_h += 0.25 * over.sum()
                cost["excess_penalty"] += 0.25 * beta * over.sum()
    if excess_h > 0:
        permitted.append(f"{excess_h:.2f} excess empty-trailer hours charged")

    # ---------------- 5. cost by component, against what the plan claims ----------------
    cost["total_cost"] = sum(cost[c] for c in ["bobtail", "empty_towing", "duty_time", "move_fees",
                                               "loaded_service", "late_penalty", "excess_penalty"])
    for c, val in cost.items():
        if c in claim.index and abs(float(claim[c]) - val) > 0.01:
            V.append(f"cost '{c}': plan claims {float(claim[c]):.2f}, checker computes {val:.2f}")
    return {"verdict": "ACCEPTED" if not V else "REJECTED", "violations": V, "permitted": permitted,
            "cost": dict(cost), "km": dict(km), "excess_hours": excess_h,
            "late_hours": sum(max(0.0, stime[j][0] - (24 * wb.shp.at[j, 'day'] + wb.shp.at[j, 'deadline_hour']))
                              for j in stime),
            "runtime_s": round(time.time() - t_start, 2)}




Check the main-file plan

Runs the checker on results/plan_main.xlsx: verdict, any violations, the permitted charges, and the recomputed cost by component.

In [5]:
res = check("data/project2_main.xlsx", "results/plan_main.xlsx")
print(res["verdict"], f"({res['runtime_s']} s)")
for v in res["violations"][:20]:
    print("  VIOLATION:", v)
for p in res["permitted"][:10]:
    print("  permitted:", p)
print({k: round(v, 2) for k, v in res["cost"].items()})

ACCEPTED (0.12 s)
  permitted: J012: 0.50 late hours charged at 60/h
  permitted: J080: 1.25 late hours charged at 80/h
  permitted: J126: 2.25 late hours charged at 100/h
  permitted: J134: 2.00 late hours charged at 80/h
  permitted: 176.25 excess empty-trailer hours charged
{'bobtail': np.float64(29020.27), 'empty_towing': np.float64(24317.51), 'duty_time': np.float64(19945.0), 'move_fees': np.float64(1567.0), 'late_penalty': 515.0, 'loaded_service': 86815.0, 'excess_penalty': np.float64(3525.0), 'total_cost': np.float64(165704.78)}


Checker tests

Builds valid and deliberately broken copies of the plans. Simplified file: a valid plan, a double-booked tractor, a missed deadline, a yard over capacity. Main file: a correctly charged late drop (accepted), and a late drop with an uncharged penalty or an unchanged release, a yard over capacity despite paying the excess penalty, and a trailer hooked at the wrong yard (all rejected).

In [6]:
"""
CHECKER TESTS (owner: Sabina)

Shows the checker accepting valid plans and rejecting broken ones. Each broken plan
is a copy of a valid plan with one deliberate change.
Simplified file: valid plan -> ACCEPT; double-booked tractor, missed deadline, yard
over capacity -> REJECT.
Main file: valid plan with charged excess -> ACCEPT; a late drop charged correctly
-> ACCEPT; the same late drop with the release left unchanged, or with no penalty
-> REJECT; a yard over capacity even with the excess paid -> REJECT; a trailer
hooked where it is not -> REJECT.

Run: python code/checker_tests.py data results
"""
import sys, os, shutil, copy
import pandas as pd

# The tests (not the checker) use the model's helpers to build realistic broken plans.



def read(path):
    p = pd.read_excel(path, sheet_name=None)
    for k, df in p.items():                    # numbers as floats, so times can be shifted
        for c in df.columns:
            if pd.api.types.is_integer_dtype(df[c]):
                df[c] = df[c].astype(float)
    return {"tours": p["tours"].set_index("tour_id"), "moves": p["moves"].set_index("move_id"),
            "shipments": p["shipments"].set_index("shipment_id"),
            "summary": p["summary"].set_index(p["summary"].columns[0])["value"]}


def write(plan, path):
    with pd.ExcelWriter(path) as xw:
        plan["tours"].to_excel(xw, sheet_name="tours")
        plan["moves"].to_excel(xw, sheet_name="moves")
        plan["shipments"].to_excel(xw, sheet_name="shipments")
        plan["summary"].to_frame("value").to_excel(xw, sheet_name="summary")


def recharge(data, plan, path):
    """Write the plan, let the checker price it, and store those prices as the plan's claim,
    so a test plan is charged correctly and only the deliberate change can be rejected."""
    write(plan, path)
    res = check(data, path)
    for c, v in res["cost"].items():
        plan["summary"][c] = v
    write(plan, path)


def shift_tour(plan, r, hours):
    """Move tour r and all its moves `hours` later."""
    t = plan["tours"]
    for c in ["depart", "return"]:
        t.at[r, c] += hours
    m = plan["moves"]
    sel = m["tour_id"] == r
    for c in ["hook", "drop_start", "drop_end"]:
        m.loc[sel, c] += hours


def retime_shipment(inst, plan, j, ready):
    t = shipment_times(inst, j, ready)
    s = plan["shipments"]
    rate = inst["shipments"].at[j, "late_penalty_cad_per_hour"]
    s.at[j, "ready"] = ready
    s.at[j, "late_hours"] = t["late_hours"]
    s.at[j, "late_penalty"] = t["late_hours"] * rate
    for c in ["load_start", "loaded_depart", "service_start", "release"]:
        s.at[j, c] = t[c]


def delivering_move(s, m, j):
    """The move that brought shipment j's trailer, or None. Gurobi may label a shipment "own stock"
    even when its trailer was delivered by a move (same cost either way), so look it up by trailer."""
    if s.at[j, "covered_by"] != "own stock":
        return s.at[j, "covered_by"]
    c = m[(m["trailer"] == s.at[j, "trailer"]) & (m["drop_end"] <= s.at[j, "ready"] + 1e-9)]
    return c["drop_end"].idxmax() if len(c) else None


def run(name, expect, data, path, rows):
    res = check(data, path)
    ok = (res["verdict"] == "ACCEPTED") == (expect == "ACCEPT")
    reason = res["violations"][0] if res["violations"] else ("; ".join(res["permitted"][:2]) or "no violation")
    rows.append({"test": name, "expected": expect, "checker": res["verdict"], "pass": "yes" if ok else "NO",
                 "reason given by the checker": reason, "violations": len(res["violations"])})


def simplified_tests(data, valid, tmp):
    inst = load_instance(data)
    rows = []
    run("Valid plan from the model", "ACCEPT", data, valid, rows)

    # double-booked tractor: give one tour to another tractor of the same depot that already drives that day
    p = read(valid); t = p["tours"]
    pair = None
    for d, g in t.groupby("day"):
        for r1 in g.index:
            for r2 in g.index:
                if r1 != r2 and t.at[r1, "depot"] == t.at[r2, "depot"]:
                    pair = (r1, r2); break
            if pair: break
        if pair: break
    r1, r2 = pair
    k2 = t.at[r2, "tractor"]
    t.at[r1, "tractor"] = k2
    p["moves"].loc[p["moves"]["tour_id"] == r1, "tractor"] = k2
    recharge(data, p, f"{tmp}/t_double.xlsx")
    run(f"Tour {r1} given to {k2}, which already drives that day", "REJECT", data, f"{tmp}/t_double.xlsx", rows)

    # missed deadline: the move that delivers a shipment's empty runs one day later
    p = read(valid); s = p["shipments"]; t = p["tours"]; m = p["moves"]
    busy = set(zip(t["tractor"], t["day"]))
    def free_next_day(mv):
        r = m.at[mv, "tour_id"]
        return (t.at[r, "tractor"], t.at[r, "day"] + 1) not in busy
    # the move must drop less than 24 h before the deadline, so one day later really is too late
    j, mv = next((j, mv) for j in s.index for mv in [delivering_move(s, m, j)]
                 if mv is not None and free_next_day(mv) and m.at[mv, "drop_end"] + 24.0 > s.at[j, "deadline"])
    r = m.at[mv, "tour_id"]
    shift_tour(p, r, 24.0); p["tours"].at[r, "day"] += 1
    retime_shipment(inst, p, j, p["moves"].at[mv, "drop_end"])
    recharge(data, p, f"{tmp}/t_late.xlsx")
    run(f"Move delivering {j}'s empty runs one day later", "REJECT", data, f"{tmp}/t_late.xlsx", rows)

    # yard over capacity: test copy of the workbook with one yard's capacity below what the plan uses
    wb = pd.read_excel(data, sheet_name=None)
    res = check(data, valid)
    g = "F03"
    wb["facilities"].loc[wb["facilities"]["facility_id"] == g, "trailer_capacity"] = 3
    with pd.ExcelWriter(f"{tmp}/wb_cap3.xlsx") as xw:
        for k, v in wb.items():
            v.to_excel(xw, sheet_name=k, index=False)
    run(f"Valid plan, {g} capacity lowered to 3 in a test copy", "REJECT", f"{tmp}/wb_cap3.xlsx", valid, rows)
    return pd.DataFrame(rows)


def main_tests(data, valid, tmp):
    inst = load_instance(data)
    rows = []
    run("Valid plan from the model (excess charged)", "ACCEPT", data, valid, rows)

    # a late drop, charged correctly: shift a delivering tour later until it drops after the deadline
    p = read(valid); s = p["shipments"]; m = p["moves"]; t = p["tours"]
    found = None
    for need in (1.0, 0.25):                       # prefer a drop at least 1 h late
      if found:
        break
      for j in s.index:
          mv = delivering_move(s, m, j)
          if mv is None or (m["tour_id"] == m.at[mv, "tour_id"]).sum() > 1:
              continue
          r = m.at[mv, "tour_id"]; k = t.at[r, "tractor"]; row = inst["tractors"].loc[k]
          D = s.at[j, "deadline"]
          for extra in [x * 0.25 for x in range(1, 49)]:
              dep = t.at[r, "depart"] + extra
              res = follow_tour(inst, row, int(t.at[r, "day"]), dep, [(m.at[mv, "from"], m.at[mv, "to"])])
              if res is None:
                  break
              mvv, ret, drive, bob, tow = res
              if mvv[0][4] >= D + need:
                  found = (j, mv, r, dep, res, row); break
          if found:
              break
    j, mv, r, dep, (mvv, ret, drive, bob, tow), row = found
    t.at[r, "depart"], t.at[r, "return"], t.at[r, "duty_hours"] = dep, ret, ret - dep
    for c, v in price(row, dep, ret, bob, tow, 1).items():
        t.at[r, c] = v
    m.at[mv, "hook"], m.at[mv, "drop_start"], m.at[mv, "drop_end"] = mvv[0][2], mvv[0][3], mvv[0][4]
    good_late = copy.deepcopy(p)
    retime_shipment(inst, good_late, j, mvv[0][4])
    recharge(data, good_late, f"{tmp}/m_late_ok.xlsx")
    late_h = good_late["shipments"].at[j, "late_hours"]
    run(f"{j}'s empty dropped {late_h:.2f} h late, penalty and later release charged", "ACCEPT", data,
        f"{tmp}/m_late_ok.xlsx", rows)

    # same late drop, but the release (and loading) left at the on-time values
    bad = copy.deepcopy(good_late)
    for c in ["load_start", "loaded_depart", "service_start", "release"]:
        bad["shipments"].at[j, c] = read(valid)["shipments"].at[j, c]
    write(bad, f"{tmp}/m_late_norelease.xlsx")
    run(f"Same late drop, release left at its on-time value", "REJECT", data, f"{tmp}/m_late_norelease.xlsx", rows)

    # same late drop, no penalty charged
    bad = copy.deepcopy(good_late)
    bad["shipments"].at[j, "late_hours"] = 0.0; bad["shipments"].at[j, "late_penalty"] = 0.0
    bad["summary"]["late_penalty"] = 0.0
    bad["summary"]["total_cost"] -= good_late["shipments"].at[j, "late_penalty"]
    write(bad, f"{tmp}/m_late_nocharge.xlsx")
    run(f"Same late drop, no late penalty charged", "REJECT", data, f"{tmp}/m_late_nocharge.xlsx", rows)

    # over capacity, excess paid: unused tractors bring idle trailers into one yard on the same day
    p = read(valid); t = p["tours"]; m = p["moves"]
    used_tr = set(m["trailer"]) | set(p["shipments"]["trailer"])
    idle = [(tr, row["facility_id"]) for tr, row in inst["trailers"].iterrows() if tr not in used_tr]
    target, day = "F13", 1                                   # a 24-hour buffer yard
    busy = set(zip(t["tractor"], t["day"]))
    added = 0
    for tr, f in idle:
        if f == target:
            continue
        for k, row in inst["tractors"].iterrows():
            if (k, day) in busy:
                continue
            res = follow_tour(inst, row, day, 24 * day + row["shift_start_hour"], [(f, target)])
            if res is None:
                continue
            mvv, ret, drive, bob, tow = res
            rid = f"X{added:03d}"
            t.loc[rid] = {"tractor": k, "driver": row["driver_name"], "depot": row["home_depot_id"], "day": day,
                          "depart": 24 * day + row["shift_start_hour"], "return": ret, "drive_hours": drive,
                          "duty_hours": ret - 24 * day - row["shift_start_hour"], "bobtail_km": bob,
                          "towing_km": tow, "n_moves": 1,
                          **price(row, 24 * day + row["shift_start_hour"], ret, bob, tow, 1)}
            m.loc[f"{rid}-1"] = {"tour_id": rid, "seq": 1, "tractor": k, "trailer": tr, "from": f, "to": target,
                                 "hook": mvv[0][2], "drop_start": mvv[0][3], "drop_end": mvv[0][4],
                                 "covers_shipment": ""}
            busy.add((k, day)); added += 1
            break
        if added >= 16:
            break
    recharge(data, p, f"{tmp}/m_overcap.xlsx")
    run(f"{added} idle trailers sent to {target} on Day {day}, excess penalty paid", "REJECT", data,
        f"{tmp}/m_overcap.xlsx", rows)

    # trailer hooked where it is not
    p = read(valid); m = p["moves"]
    mv = m.index[0]
    other = next(tr for tr, row in inst["trailers"].iterrows()
                 if row["facility_id"] != m.at[mv, "from"] and tr not in set(m["trailer"]))
    m.at[mv, "trailer"] = other
    for j in p["shipments"].index:
        if delivering_move(p["shipments"], m, j) == mv:
            p["shipments"].at[j, "trailer"] = other
    write(p, f"{tmp}/m_wrongtrailer.xlsx")
    run(f"Move {mv} hooks {other}, which is at another yard", "REJECT", data, f"{tmp}/m_wrongtrailer.xlsx", rows)
    return pd.DataFrame(rows)




Run the tests

Each row shows the expected verdict, the checker’s verdict, whether they match, and the reason the checker gave. Results are also saved to results/checker_tests.xlsx.

In [7]:
data_dir, res_dir = "data", "results"
tmp = f"{res_dir}/checker_tests"
os.makedirs(tmp, exist_ok=True)
out = {}
if os.path.exists(f"{res_dir}/plan_simplified.xlsx"):
    out["simplified"] = simplified_tests(f"{data_dir}/project2_simplified.xlsx", f"{res_dir}/plan_simplified.xlsx", tmp)
if os.path.exists(f"{res_dir}/plan_main.xlsx"):
    out["main"] = main_tests(f"{data_dir}/project2_main.xlsx", f"{res_dir}/plan_main.xlsx", tmp)
with pd.ExcelWriter(f"{res_dir}/checker_tests.xlsx") as xw:
    for k, v in out.items():
        v.to_excel(xw, sheet_name=k, index=False)
        print(f"\n== {k}\n" + v.to_string(index=False))


== simplified
                                                    test expected  checker pass                                              reason given by the checker  violations
                               Valid plan from the model   ACCEPT ACCEPTED  yes                                                             no violation           0
Tour R002175 given to T07, which already drives that day   REJECT REJECTED  yes                             tractor T07 double-booked: 2 shifts on day 0           1
         Move delivering J019's empty runs one day later   REJECT REJECTED  yes                     J019: missed deadline, ready at 108.75 > deadline 88           1
    Valid plan, F03 capacity lowered to 3 in a test copy   REJECT REJECTED  yes F03: yard holds 4 trailers at 39.25 > capacity 3 (10 quarter hours over)           1

== main
                                                               test expected  checker pass                                                             